## QuTip Tutorial :)

Today we are *drumrolls please!!* simulating...

A single qubit rotation!! Tada!!


The hamiltonian of this is...

$H = \frac{\omega}{2}\sigma_x$

Initialize the state as: $|0>$

In [ ]:
from qutip import *
import numpy as np
import matplotlib.pyplot as plt



In [ ]:
# Set parameters

omega = 2 * np.pi #-> angular frequency of the drive
t_max = 2 # -> total time evolution
n_steps = 200 


In [ ]:
# Set operators and the space
ket0 = basis(2,0) # 2 dimensional ket or column vector set in the 0 state (1,0) for our space
ket1 = basis(2,1)

sx = sigmax()
sz = sigmaz() # -> these are our operators in our space (2x2 of course! :)

# Set the hamiltonian

H = 0.5 * omega * sx

# Initialize state
psi0 = ket0

In [ ]:
tlist = np.linspace(0,t_max,n_steps)


result = mesolve(H,psi0, tlist, [], [sz]) #-> this gives <sz>(t)


In [ ]:
expect_sz = result.expect[0] # -> extracts the result <sz>(t)
p0 = (1 + expect_sz) * 0.5 #-> population in 0 state (sz = p0 - p1)

In [ ]:
# --- Plot ---
plt.figure()
plt.plot(tlist, p0)
plt.xlabel('Time (s)')
plt.ylabel('Population in |0>')
plt.title('Rabi oscillation under H = (ω/2) σ_x')
plt.grid(True)
plt.show()

## Example 2:
Spinning harmonic oscillator!!
$H = \frac{p^{2}}{2m} + \frac{1}{2}m\omega_ox^{2} + \frac{\omega_s}{2}\sigma_z$

No coupling so you split it up and solve both independently!! ^^

Notice how the oscillator part is consistently on the left!

$H = H_{o} \otimes I_s + I_o \otimes H_s$



$\psi(t) = (exp[-iH_ot] * \phi_o) \otimes (exp[iH_st] *\chi_s)$




**Simulating Bit**

You see it here!!

$H = H_o \otimes I_s + I_o \otimes H_s$

$H_o = \frac{p^{2}}{2m} + \frac{1}{2}m\omega_ox^{2}$

$H_s = \frac{\omega_s}{2}\sigma_z$

Know the energies too!

$E_n = \omega(n + \frac{1}{2})$

$E_s = \pm \frac {\omega}{2}$

yuppies we're gonna do this!!


Optional tweaks:

Add a coupling term and see what it does ^^

$H_{coup} = gx \otimes \sigma_x$

Start in different inital states and see what happens

Ex: $\chi_s(0) = |\uparrow>$

Ex: Set $\omega_{spin} = \omega_{oscillation}$ and see what happens

In [ ]:
# Set parameters

N = 40
omega_osc = 1.0
omega_s = 1.5
t_max = 10.0
n_steps = 800
tlist = np.linspace(0,t_max,n_steps)

# Coupling parameter 
g = 1

In [ ]:
# make the oscillator operators!!
a = destroy(N) # lower ladder operator or annihilation operator
n_op = a.dag() * a 
x_op = (a + a.dag()) / np.sqrt(2) # -> dimensionlessrepresentation of position operator in terms of annihilation 
p_op = -1j * (a - a.dag()) / np.sqrt(2) #-> same for momentum

H_osc = omega_osc * 0.5 * (x_op @ x_op + p_op @ p_op)

H_osc

In [ ]:
# Make the spin operators!!
sx = sigmax()
sy = sigmay()
sz = sigmaz()
H_s = 0.5 * omega_s * sz
H_s

In [ ]:
# Adding in optional interaction term!

H_couple = g * tensor(x_op, sx) 

In [ ]:
# Full tensor system via kronecker product
H = tensor(H_osc, qeye(2)) + tensor(qeye(N), H_s)
H_coupled = H + H_couple

In [ ]:
# Make full tensor sys  cont.
X_full = tensor(x_op, qeye(2))
P_full = tensor(p_op, qeye(2))
SX_full = tensor(qeye(N), sx)
SY_full = tensor(qeye(N), sy)
SZ_full = tensor(qeye(N), sz)


In [ ]:
# Initialize system!!
alpha = 2 # -> coherent amplitude (might recall :))

psi_osc0 = coherent(N, alpha)
psi_spin0 = (basis(2,0) + basis(2,1)).unit() #-> normalized version of column vector [1,1] (the +|x> eigenvector of s_x)
psi_spin1 = (basis(2,1))
# Full initial state:
psi0 = tensor(psi_osc0,psi_spin0)
psi1 = tensor(psi_osc0, psi_spin1)


In [ ]:
# Solve time!!
e_ops = [X_full, P_full, SX_full, SY_full, SZ_full] # observables to measure :>
result = mesolve(H, psi0, tlist, [], e_ops)

In [ ]:
x_exp = result.expect[0]
p_exp = result.expect[1]
sx_exp = result.expect[2]
sy_exp = result.expect[3]
sz_exp = result.expect[4]

In [ ]:
# --- Plot oscillator position and spin expectation values ---
plt.figure(figsize=(10,6))

plt.subplot(2,1,1)
plt.plot(tlist, x_exp, label=r'$\langle x\rangle$')
plt.plot(tlist, p_exp, label=r'$\langle p\rangle$', alpha=0.7)
plt.legend()
plt.ylabel('Oscillator (dimensionless)')
plt.title('Oscillator and spin expectations — uncoupled HO + spin')

plt.subplot(2,1,2)
plt.plot(tlist, sx_exp, label=r'$\langle\sigma_x\rangle$')
plt.plot(tlist, sy_exp, label=r'$\langle\sigma_y\rangle$')
plt.plot(tlist, sz_exp, label=r'$\langle\sigma_z\rangle$')
plt.legend()
plt.xlabel('Time')
plt.ylabel('Spin expectations')

plt.tight_layout()
plt.show()

In [ ]:
# Solve for coupled sys
# Solve time!!
e_ops_c = [X_full, P_full, SX_full, SY_full, SZ_full] # observables to measure :>
result_c = mesolve(H_coupled, psi0, tlist, [], e_ops_c)
x_exp_c = result_c.expect[0]
p_exp_c = result_c.expect[1]
sx_exp_c = result_c.expect[2]
sy_exp_c = result_c.expect[3]
sz_exp_c = result_c.expect[4]


In [ ]:
# --- Plot oscillator position and spin expectation values ---
plt.figure(figsize=(10,6))

plt.subplot(2,1,1)
plt.plot(tlist, x_exp_c, label=r'$\langle x\rangle$')
plt.plot(tlist, p_exp_c, label=r'$\langle p\rangle$', alpha=0.7)
plt.legend()
plt.ylabel('Oscillator (dimensionless)')
plt.title(f'Oscillator and spin expectations —  HO + spin + coupled x*s_x term with weight {g}')

plt.subplot(2,1,2)
plt.plot(tlist, sx_exp_c, label=r'$\langle\sigma_x\rangle$')
plt.plot(tlist, sy_exp_c, label=r'$\langle\sigma_y\rangle$')
plt.plot(tlist, sz_exp_c, label=r'$\langle\sigma_z\rangle$')
plt.legend()
plt.xlabel('Time')
plt.ylabel('Spin expectations')

plt.tight_layout()
plt.show()

# Observations!!
When setting $\omega_{spin} = \omega_{oscillation}$, the measurements,

$<x>(t)$ and $<\sigma_x>(t)$ are perfectly in phase

$<p>(t)$ and $<\sigma_y>(t)$ as well!

When adding the coupling term $gx \otimes \sigma_x$, we observe this!

g = 0.25 -> $x,p,\sigma_x$ basically unperturbed

g = 2 -> only $x,p$ unperturbed, slight damping

g = 10 -> everyone messed up! x,p still exhibit wave pattern. x more perturbed than p

## Example 3

Cavity QED Model

$H = \frac{\omega_q}{2}\sigma_z + \omega_ca^{\dagger}a + g(a^{\dagger}\sigma_{-}+a\sigma_+) $

In [ ]:
# Set paraemeters

omega_q = 1.5
omega_c = 1
g = 0.5
N = 45
n_steps = 10**3
n_steps
t_max = 10
tlist = np.linspace(0,t_max,n_steps)


alpha = 2

In [ ]:
# Make operators
a = destroy(N)

# Spin operators
sx = sigmax()
sy = sigmay()
sz = sigmaz()
sminus = sx - 1j*sy
splus = sx + 1j*sy

In [ ]:
# Make hamiltonian components

# Spin hamiltonian
H_s = 0.5 * omega_c * sz

# Harmonic hamiltonian
H_o = omega_c * a.dag() *a

# Couple term
H_coup = g * (tensor(a.dag(),sminus) + tensor(a,splus))

In [ ]:
# Make overall hamiltonian
H = tensor(H_o,qeye(2)) + tensor(qeye(N), H_s) + H_coup

In [ ]:
# Initialize state

# Spin state
spin0 = basis(2,0)

# Harmonic state
harmonic0 = coherent(N, alpha)

# Overall state:

psi0 = tensor(harmonic0, spin0)



In [ ]:
x_op = (a + a.dag()) / np.sqrt(2)
p_op = -1j * (a - a.dag()) / np.sqrt(2)

In [ ]:
# Observables to measure:
# Make full tensor sys  cont.
X_full = tensor(x_op, qeye(2))
P_full = tensor(p_op, qeye(2))
SX_full = tensor(qeye(N), sx)
SY_full = tensor(qeye(N), sy)
SZ_full = tensor(qeye(N), sz)
Sminus_full = tensor(qeye(N), sminus)
Splus_full = tensor(qeye(N), splus)


In [ ]:
e_ops = [X_full, P_full, SX_full, SY_full, SZ_full, Sminus_full, Splus_full] # observables to measure :>
result = mesolve(H, psi0, tlist, [], e_ops)

In [ ]:
x_exp = result.expect[0]
p_exp = result.expect[1]
sx_exp = result.expect[2]
sy_exp = result.expect[3]
sz_exp = result.expect[4]
sm_exp = result.expect[5]
sp_exp = result.expect[6]

In [ ]:
H_o.eigenenergies()

In [ ]:
# --- Plot oscillator position and spin expectation values ---
plt.figure(figsize=(10,6))

plt.subplot(2,1,1)
plt.plot(tlist, x_exp, label=r'$\langle x\rangle$')
plt.plot(tlist, p_exp, label=r'$\langle p\rangle$', alpha=0.7)
plt.legend()
plt.ylabel('Oscillator (dimensionless)')
plt.title(f'Oscillator and spin expectations (Cavity QED Model) —  HO + spin + coupled term of weight {g}')

plt.subplot(2,1,2)
plt.plot(tlist, sx_exp, label=r'$\langle\sigma_x\rangle$')
plt.plot(tlist, sy_exp, label=r'$\langle\sigma_y\rangle$')
plt.plot(tlist, sz_exp, label=r'$\langle\sigma_z\rangle$')

plt.xlabel('Time')
plt.ylabel('Spin expectations')

plt.tight_layout()
plt.show()

In [ ]:
a = destroy(20)
print(a.data)

In [ ]:
b = Bloch()

In [ ]:
b.render()
b.show()